# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShubhamSnSharma/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["PYTHONHASHSEED"] = "0"

In [2]:
!pip install scikit-learn==1.6.1 --quiet

In [3]:
# Imports
import duckdb
from getpass import getpass

# Authenticate with Hugging Face
token = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

Enter your Hugging Face READ token: ··········


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



---

I chose Logistic Regression as the primary model for this capstone because it provides a transparent baseline for learning from structured tabular data while remaining easy to interpret.

My Week 4 baseline was a hand-written rule that combined GSC impressions, GSC average position, and GA4 pageviews into a simple priority score using fixed thresholds. Logistic Regression extends this idea by learning the contribution of each feature directly from the data instead of relying on manually chosen decision boundaries.

This model is well suited to the structured tabular nature of the warehouse dataset and provides well-calibrated probability scores that can be used to rank pages for review using the same Precision@50 metric as the Week 4 baseline. It also allows the effect of each feature to be interpreted through its learned coefficients, making it easy to understand why pages receive higher or lower predicted risk.

I selected Logistic Regression because the goal of this assignment is to determine whether a simple, interpretable learned model can improve on the Week 4 baseline using the same data, the same validation strategy, and the same evaluation metric. More complex models such as Random Forest can be evaluated afterwards to determine whether the additional complexity produces a meaningful improvement over this transparent baseline.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


---

I used a client-aware and time-aware validation strategy because the goal is to predict which pages are likely to decline for clients the model has not seen before.

The modeling dataset is built using features aggregated from February through April 2026, while the target is defined using page performance during May 2026. This creates a clear separation between the feature window and the target window, ensuring that only information available before the prediction date is used by the model and preventing target leakage.

The train and test sets are split by `client_hash_id` rather than by random rows. This prevents pages from the same client appearing in both sets, reducing the risk that the model learns client-specific behaviour instead of patterns that generalize across different websites.

Both the Week 4 baseline and the Logistic Regression model are evaluated on the same held-out clients using the same Precision@50 metric. Using the same data, the same split, and the same evaluation metric provides a fair comparison between the hand-written baseline and the learned model.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

---

### Baseline implementation

The Week 4 baseline heuristic (`HIGH_VISIBILITY`: impressions >= 10, `MID_RANKING`: average position > 10, `HAS_TRAFFIC`: pageviews >= 1) was originally defined on daily page observations. To preserve the original rule while changing the decision grain to page-level predictions, I evaluated the three-signal rule on every daily row in April 2026 and then averaged the resulting daily baseline scores for each page (`client_hash_id`, `content_hash_id`). This preserves the original thresholds without reinterpreting them as monthly averages, allowing a fair comparison between the Week 4 heuristic and the learned model.

> **Deterministic Tie-Breaking Policy:** Both the baseline and Logistic Regression were evaluated on the same held-out test pages using the same Precision@50 metric. When identical scores occurred, pages were ordered deterministically using `(score DESC, content_hash_id ASC)` to ensure reproducible top-K rankings.

> **Note on methodology:** this notebook's Week 4 rule reproduction (Precision@50 = 0.22) scores the rule per day across April, then averages those daily scores per page. A later investigation, `ml08_modeling_investigation.ipynb`, reproduces the same original rule using monthly-aggregate scoring instead (Precision@50 = 0.26), to match the page-level grain used across that investigation's full model comparison. Both are legitimate adaptations of the same rule from `w04_baseline_score.ipynb`, evaluated two different ways at two different points in the project. `ml08`'s numbers are the ones cited going forward in the portfolio, resume, and any published writeup — this notebook is preserved as the original Week 5 submission.

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# -------------------------------------------------------------------------
# Step 1: Query Hugging Face Warehouse & Aggregate Page-Level Metrics
# -------------------------------------------------------------------------
# Feature window: Feb 1 – Apr 30, 2026
# Target window: May 1 – May 31, 2026
# Eligibility: Feb–Apr impressions >= 1000 AND April clicks >= 10
# -------------------------------------------------------------------------

modeling_df = con.sql(f"""
WITH daily_facts AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        gsc_avg_position,
        COALESCE(ga4_pageviews, 0) AS ga4_pageviews,
        COALESCE(ga4_engaged_sessions, 0) AS ga4_engaged_sessions,
        COALESCE(client_has_gsc, FALSE) AS client_has_gsc,
        COALESCE(client_has_ga4, FALSE) AS client_has_ga4,
        COALESCE(gsc_data_available, FALSE) AS gsc_data_available,
        COALESCE(ga4_data_available, FALSE) AS ga4_data_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month IN ('2026-02', '2026-03', '2026-04', '2026-05')
),
aggregated_pages AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature window aggregations (Feb - Apr 2026)
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_impressions ELSE 0 END) AS gsc_impressions_feb_apr,
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_clicks ELSE 0 END) AS gsc_clicks_feb_apr,
        SUM(CASE WHEN month = '2026-04' THEN gsc_impressions ELSE 0 END) AS gsc_impressions_apr,
        SUM(CASE WHEN month = '2026-04' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_apr,
        AVG(CASE WHEN month = '2026-04' THEN gsc_avg_position ELSE NULL END) AS gsc_avg_position_apr,

        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_feb_apr,
        SUM(CASE WHEN month = '2026-04' THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_apr,
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions_feb_apr,

        MAX(client_has_gsc) AS client_has_gsc,
        MAX(client_has_ga4) AS client_has_ga4,
        MAX(gsc_data_available) AS gsc_data_available,
        MAX(ga4_data_available) AS ga4_data_available,

        -- Target window aggregation (May 2026) - ISOLATED FOR TARGET ONLY
        SUM(CASE WHEN month = '2026-05' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_may

    FROM daily_facts
    GROUP BY client_hash_id, content_hash_id
)
SELECT *
FROM aggregated_pages
WHERE gsc_impressions_feb_apr >= 1000
  AND gsc_clicks_apr >= 10
""").df()

import datetime
pull_timestamp = datetime.datetime.now().isoformat()
print("Data pulled at:", pull_timestamp)

modeling_df = modeling_df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

# -------------------------------------------------------------------------
# Step 2: Create Supervised Binary Target (is_declining_label)
# -------------------------------------------------------------------------
# Target: May 2026 clicks < 0.80 * April 2026 clicks (>=20% decay)
# -------------------------------------------------------------------------

modeling_df["is_declining_label"] = (
    modeling_df["gsc_clicks_may"] < (0.80 * modeling_df["gsc_clicks_apr"])
).astype(int)

# Impute missing average position (0.0 indicates no search rank recorded)
modeling_df["gsc_avg_position_apr"] = modeling_df["gsc_avg_position_apr"].fillna(0.0)

# -------------------------------------------------------------------------
# Step 3: Feature Engineering & Data Validation Checks
# -------------------------------------------------------------------------
# Log-transform right-skewed count features to stabilize variance and
# reduce the influence of extreme outlier traffic values on linear coefficients.
# -------------------------------------------------------------------------

modeling_df["log_gsc_impressions_feb_apr"] = np.log1p(modeling_df["gsc_impressions_feb_apr"])
modeling_df["log_gsc_clicks_apr"] = np.log1p(modeling_df["gsc_clicks_apr"])
modeling_df["log_ga4_pageviews_feb_apr"] = np.log1p(modeling_df["ga4_pageviews_feb_apr"])
modeling_df["log_ga4_engaged_sessions_feb_apr"] = np.log1p(modeling_df["ga4_engaged_sessions_feb_apr"])

feature_cols = [
    "log_gsc_impressions_feb_apr",
    "log_gsc_clicks_apr",
    "gsc_avg_position_apr",
    "log_ga4_pageviews_feb_apr",
    "log_ga4_engaged_sessions_feb_apr",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

X = modeling_df[feature_cols].astype(float)
y = modeling_df["is_declining_label"].astype(int)
groups = modeling_df["client_hash_id"].astype(str)

print("=== Data Validation & Sanity Checks ===")
print(f"Modeling DataFrame shape: {modeling_df.shape}")
print(f"Missing values check in X:\n{X.isnull().sum().to_dict()}")
print(f"Target class counts:\n{y.value_counts().to_dict()}")
print(f"Base rate (declining proportion): {y.mean():.4f}")

# -------------------------------------------------------------------------
# Step 4: Client-Aware Train/Test Split via GroupShuffleSplit
# -------------------------------------------------------------------------
# Using GroupShuffleSplit ensures 100% zero client overlap between train and test sets.
# -------------------------------------------------------------------------

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = groups.iloc[train_idx].nunique()
test_clients = groups.iloc[test_idx].nunique()

print(f"\n=== GroupShuffleSplit Summary ===")
print(f"Train set: {len(X_train):,} rows across {train_clients} clients")
print(f"Test set:  {len(X_test):,} rows across {test_clients} clients")

# -------------------------------------------------------------------------
# Step 5: Compute Week 4 Baseline Score (Daily Computed -> Page Averaged)
# -------------------------------------------------------------------------
# The original Week 4 rule is evaluated on each daily row in April 2026:
#
# daily_baseline_score =
#     (gsc_impressions >= 10)
#   + (gsc_avg_position > 10)
#   + (ga4_pageviews >= 1)
#
# These daily scores are then averaged per page to preserve the original
# Week 4 rule while matching the page-level decision grain used in Week 5.
# -------------------------------------------------------------------------

baseline_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    AVG(
        (
            CASE WHEN COALESCE(gsc_impressions,0) >= 10 THEN 1 ELSE 0 END +
            CASE WHEN gsc_avg_position > 10 THEN 1 ELSE 0 END +
            CASE WHEN COALESCE(ga4_pageviews,0) >= 1 THEN 1 ELSE 0 END
        )
    ) AS baseline_score

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)

WHERE month = '2026-04'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

modeling_df = modeling_df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

# Merge the original Week 4 baseline into the modeling dataset
modeling_df = modeling_df.merge(
    baseline_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

modeling_df["baseline_score"] = modeling_df["baseline_score"].fillna(0.0)

# -------------------------------------------------------------------------
# Precision@K helper
# -------------------------------------------------------------------------

def precision_at_k(y_true, scores, tie_breakers=None, k=50):
    eval_df = pd.DataFrame({
        "y": list(y_true),
        "score": list(scores)
    })

    if tie_breakers is not None:
        eval_df["tie_breaker"] = list(tie_breakers)
        eval_df = eval_df.sort_values(
            ["score", "tie_breaker"],
            ascending=[False, True]
        )
    else:
        eval_df["row_id"] = np.arange(len(eval_df))
        eval_df = eval_df.sort_values(
            ["score", "row_id"],
            ascending=[False, True]
        )

    top_k = eval_df.head(min(k, len(eval_df)))

    return top_k["y"].mean()

# Evaluate baseline on the held-out test pages using original Week 4 multi-key tie-breaking
test_df = modeling_df.iloc[test_idx].copy()
test_content_ids = test_df["content_hash_id"]

test_df_baseline_sorted = test_df.sort_values(
    by=["baseline_score", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
    ascending=[False, False, False, True]
).reset_index(drop=True)
baseline_p50 = float(test_df_baseline_sorted.head(50)["is_declining_label"].mean())

# -------------------------------------------------------------------------
# Step 6 & 7: Train Logistic Regression & Predict Test Probabilities
# -------------------------------------------------------------------------

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced",random_state=42,max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
test_probabilities = lr_pipeline.predict_proba(X_test)[:, 1]

# -------------------------------------------------------------------------
# Step 8 & 9: Compute Primary Metric & Display Focused Comparison Table
# -------------------------------------------------------------------------

model_p50 = precision_at_k(y_test, test_probabilities, tie_breakers=test_content_ids, k=50)

# Primary comparison table focused strictly on Precision@50
comparison_df = pd.DataFrame([
    {
        "Approach": "Week 4 Baseline Rule",
        "Precision@50": f"{baseline_p50:.4f}"
    },
    {
        "Approach": "Logistic Regression (W5)",
        "Precision@50": f"{model_p50:.4f}"
    }
])

print("\n=== Primary Evaluation Comparison Table ===")
print(comparison_df.to_string(index=False))

print(f"\nImprovement over baseline: {model_p50 - baseline_p50:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data pulled at: 2026-08-16T09:32:27.676051
=== Data Validation & Sanity Checks ===
Modeling DataFrame shape: (16513, 20)
Missing values check in X:
{'log_gsc_impressions_feb_apr': 0, 'log_gsc_clicks_apr': 0, 'gsc_avg_position_apr': 0, 'log_ga4_pageviews_feb_apr': 0, 'log_ga4_engaged_sessions_feb_apr': 0, 'client_has_gsc': 0, 'client_has_ga4': 0, 'gsc_data_available': 0, 'ga4_data_available': 0}
Target class counts:
{0: 9640, 1: 6873}
Base rate (declining proportion): 0.4162

=== GroupShuffleSplit Summary ===
Train set: 14,786 rows across 28 clients
Test set:  1,727 rows across 8 clients


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Primary Evaluation Comparison Table ===
                Approach Precision@50
    Week 4 Baseline Rule       0.2000
Logistic Regression (W5)       0.2400

Improvement over baseline: 0.0400


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

---

### Interpretation

The Logistic Regression model achieved a Precision@50 of **0.24**, compared with **0.22** for the Week 4 heuristic baseline. This indicates that the learned model provides a small improvement over the manually designed rule while using the same data, validation strategy, and evaluation metric.

The coefficient table shows that the model relies most strongly on historical GA4 pageviews, data availability, average search position, and April click volume when estimating the probability of future decline. Unlike the Week 4 baseline, which applies fixed thresholds independently to each signal, Logistic Regression learns how these features interact to produce a probability score.

Although the learned model outperformed the baseline, the improvement is modest. This suggests that the current feature set captures only part of the information needed to predict future traffic decline. Additional temporal features such as traffic trends, CTR changes, or ranking movement may improve predictive performance in future work.

These results support the conclusion that a simple learned model can outperform a transparent heuristic, but they also show that future outcome prediction remains a challenging task with the available features.

In [5]:
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lr_pipeline.named_steps["model"].coef_[0]
})

coef_df["Absolute Coefficient"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values(
    "Absolute Coefficient",
    ascending=False
).reset_index(drop=True)

print("=== Logistic Regression Coefficients ===")
display(coef_df)

print(f"\nModel Intercept: {lr_pipeline.named_steps['model'].intercept_[0]:.4f}")

=== Logistic Regression Coefficients ===


,Feature,Coefficient,Absolute Coefficient
0,log_ga4_pageviews_feb_apr,-0.939398,0.939398
1,ga4_data_available,0.706044,0.706044
2,gsc_avg_position_apr,0.321219,0.321219
3,log_gsc_clicks_apr,0.280714,0.280714
4,client_has_ga4,-0.175088,0.175088
5,log_ga4_engaged_sessions_feb_apr,0.044646,0.044646
6,log_gsc_impressions_feb_apr,-0.023094,0.023094
7,client_has_gsc,0.000000,0.000000
8,gsc_data_available,0.000000,0.000000



Model Intercept: -0.0098


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.